# Apple Detector — Kaggle Training (Synthetik-Sweep + W&B-Tracking)

**Workflow:** Daten einmal hochladen. Pro Sweep-Punkt nur `SYNTHETIC_N` ändern, Trainingsset
aus `/kaggle/input` zusammenbauen (benchmark-frei), trainieren, auf dem **eingefrorenen
Benchmark** evaluieren, und Config + Metriken nach **Weights & Biases** loggen.

> Voraussetzungen:
> 1. Repo-Commits nach GitHub gepusht (das Notebook klont von dort).
> 2. Daten-Dataset als Input angehängt.
> 3. **W&B-API-Key als Kaggle-Secret** `WANDB_API_KEY` hinterlegt
>    (Add-ons → Secrets). Free-Account: wandb.ai → User Settings → API keys.
> 4. Für den Sweep: `SYNTHETIC_N` variieren, alles andere KONSTANT, und jeden Run
>    per **"Save Version" (Save & Run All)** committen.


## Setup

In [ ]:
!git clone https://github.com/philippsnr/apple-vision.git
%cd apple-vision

In [ ]:
!cd /kaggle/working/apple-vision && git pull && uv sync --quiet

## Konfiguration — ein Sweep-Punkt

Für den Sweep **nur `SYNTHETIC_N`** variieren. Reale Basis, Seed, Augmentierung, Epochs
über den gesamten Sweep **konstant** halten.

In [ ]:
import os

# --- Pfade (KAGGLE_INPUT an deinen Dataset-Slug anpassen!) ---
KAGGLE_INPUT = '/kaggle/input/apple-vision-data'
REPO = '/kaggle/working/apple-vision'
WORK = '/kaggle/working'

# --- Sweep-Regler: NUR diese Zahl variieren (0, 100, 200, 400, 800, 'all') ---
SYNTHETIC_N = 0

# --- Reale Basis: über den GESAMTEN Sweep fix ---
REAL_SOURCES = {'minneapple': 'all', 'apple_mots': 'all', 'orchard': 'all'}
SEED = 42
VAL_RATIO = 0.1

# --- Training ---
BASE_MODEL = 'fasterrcnn_resnet50_fpn'
EPOCHS = 30
BATCH_SIZE = 4
USE_AMP    = True         # Mixed Precision auf T4: schneller + halber VRAM (bei Batch-Erhöhung nützlich)

# --- Augmentierung: über den GESAMTEN Sweep KONSTANT (sonst Confound) ---
AUG_FACTOR = 4
AUG_FLAGS = '--aug-brightness 0.3 --aug-contrast 0.3 --aug-saturation 0.2 --aug-hue 0.05 --aug-hflip 0.5'

# --- Experiment-Tracking ---
WANDB_PROJECT = 'apple-detector'
RUN_NAME = f'synth{SYNTHETIC_N}-seed{SEED}'

# --- abgeleitete Pfade ---
TRAIN_ROOT   = f'{WORK}/train_set/coco'
TRAIN_MANIFEST = f'{WORK}/train_set/training_manifest.json'
BENCH_ROOT   = f'{KAGGLE_INPUT}/benchmark/coco'
MANIFEST     = f'{KAGGLE_INPUT}/benchmark_manifest.json'
CKPT         = f'{WORK}/checkpoints/fasterrcnn_resnet50_fpn_apple_best.pth'
METRICS_JSON = f'{WORK}/checkpoints/metrics_{RUN_NAME}.json'
os.makedirs(f'{WORK}/checkpoints', exist_ok=True)
print('Sweep-Punkt:', RUN_NAME)


## Check — Input-Dataset vollständig?

In [ ]:
import os, json

ok = True
for tag in list(REAL_SOURCES) + ['synthetic']:
    ann = f'{KAGGLE_INPUT}/{tag}/coco/annotations/instances_train.json'
    if os.path.exists(ann):
        print(f'{tag:12s} train images: {len(json.load(open(ann))["images"])}')
    else:
        print(f'{tag:12s} FEHLT: {ann}'); ok = False

bench = f'{BENCH_ROOT}/annotations/instances_test.json'
print('benchmark   :', len(json.load(open(bench))['images']) if os.path.exists(bench) else 'FEHLT', 'images')
print('manifest    :', 'OK' if os.path.exists(MANIFEST) else 'FEHLT')
assert ok and os.path.exists(bench) and os.path.exists(MANIFEST), 'Input unvollständig — KAGGLE_INPUT prüfen'


## Trainingsset zusammenbauen (aus /kaggle/input, garantiert benchmark-frei)

In [ ]:
sources = ' '.join(f'--source {KAGGLE_INPUT}/{tag}/coco:{n}' for tag, n in REAL_SOURCES.items())
sources += f' --source {KAGGLE_INPUT}/synthetic/coco:{SYNTHETIC_N}'

compose = (f'cd {REPO} && uv run python scripts/build_training_set.py {sources} '
           f'--benchmark-manifest {MANIFEST} --output {TRAIN_ROOT} '
           f'--val-ratio {VAL_RATIO} --val-exclude synthetic --seed {SEED}')
print(compose)
!{compose}


## Training

In [ ]:
train = (f'cd {REPO} && MPLBACKEND=agg uv run python -m apple_vision.train '
         f'--dataset-root {TRAIN_ROOT} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers 4 '
         f'--early-stop-patience 5 {"--amp" if USE_AMP else ""} --aug-factor {AUG_FACTOR} {AUG_FLAGS} '
         f'--out-dir {WORK}/checkpoints')
print(train)
!{train}


In [ ]:
from IPython.display import display, Image
display(Image(filename=f'{WORK}/checkpoints/detector_loss.png'))


## Evaluate — eingefrorener Benchmark (für ALLE Modelle identisch)

Schreibt benannte Metriken (AP, AP50, ...) nach `METRICS_JSON` für W&B + Aggregation.

In [ ]:
evalc = (f'cd {REPO} && uv run python -m apple_vision.evaluate_coco '
         f'--dataset-root {BENCH_ROOT} --val-ann annotations/instances_test.json '
         f'--val-images images/test --checkpoint {CKPT} '
         f'--results-json {WORK}/checkpoints/coco_results_{RUN_NAME}.json '
         f'--metrics-json {METRICS_JSON}')
print(evalc)
!{evalc}


## Tracking — Weights & Biases

Loggt **Config (alle Parameter)** + **Benchmark-Metriken** + Loss-Kurve + das Checkpoint
als Artefakt. Im W&B-Web-UI: sortierbare Runs-Tabelle nach AP/AP50, Parameter als Spalten.

In [ ]:
import json, os
try:
    import wandb
except ImportError:
    !pip install -q wandb
    import wandb

# API-Key aus Kaggle-Secret
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    wandb_ok = True
except Exception as e:
    print('W&B-Login übersprungen (Secret WANDB_API_KEY fehlt?):', e)
    wandb_ok = False

if wandb_ok:
    config = {
        'synthetic_n': SYNTHETIC_N, 'real_sources': REAL_SOURCES,
        'seed': SEED, 'val_ratio': VAL_RATIO,
        'base_model': BASE_MODEL, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'aug_factor': AUG_FACTOR, 'aug_flags': AUG_FLAGS, 'amp': USE_AMP,
    }
    if os.path.exists(TRAIN_MANIFEST):
        tm = json.load(open(TRAIN_MANIFEST))
        config['train_images'] = tm['totals']['train']
        config['val_images'] = tm['totals']['val']
        config['train_boxes'] = sum(s['boxes'] for s in tm['sources'])

    run = wandb.init(project=WANDB_PROJECT, name=RUN_NAME, config=config, reinit=True)

    # Benchmark-Metriken -> summary (= sortierbare Tabellenspalten)
    metrics = json.load(open(METRICS_JSON))
    run.summary.update(metrics)
    print('AP=%.4f  AP50=%.4f' % (metrics['AP'], metrics['AP50']))

    # Loss-Kurve (per Epoche)
    import csv as _csv
    loss_csv = f'{WORK}/checkpoints/detector_metrics.csv'
    if os.path.exists(loss_csv):
        for row in _csv.DictReader(open(loss_csv)):
            wandb.log({'epoch': int(float(row['epoch'])),
                       'train_loss': float(row['train_loss']),
                       'val_loss': float(row['val_loss'])})

    # Checkpoint als Artefakt (an diesen Run gekoppelt)
    if os.path.exists(CKPT):
        art = wandb.Artifact(f'detector-{RUN_NAME}', type='model', metadata=config)
        art.add_file(CKPT)
        run.log_artifact(art)

    run.finish()
    print('W&B:', run.url)


## Visualize — Predictions auf dem Benchmark

In [ ]:
vis = (f'cd {REPO} && MPLBACKEND=agg uv run python -m apple_vision.visualize_detections '
       f'--checkpoint {CKPT} --dataset-root {BENCH_ROOT} --split test '
       f'--score-threshold 0.5 --n 8 --out-dir {WORK}/quickplots/detections')
print(vis)
!{vis}


In [ ]:
from IPython.display import display, Image
from pathlib import Path
for p in sorted(Path(f'{WORK}/quickplots/detections').glob('*.png'))[:4]:
    display(Image(filename=str(p)))


## (Nach dem Sweep) Bestes Modell in die Kaggle-Model-Registry

Erst ausführen, wenn ein Sieger feststeht — nicht pro Sweep-Punkt. Legt das gewählte
Checkpoint versioniert ab, damit es die 3D-Pipeline wiederverwenden kann.
Handle-Format: `<username>/<model-slug>/<framework>/<variation>`.

In [ ]:
# import kagglehub, shutil, os
# upload_dir = f'{WORK}/model_upload'; os.makedirs(upload_dir, exist_ok=True)
# shutil.copy(CKPT, upload_dir)
# kagglehub.model_upload(
#     handle='philippstaudinger/apple-detector/pyTorch/faster-rcnn-r50',
#     local_model_dir=upload_dir,
#     version_notes=RUN_NAME,
# )
